# DL Cluster

Ноутбук проверяет cluster meta-признаки для DL-моделей.
Кластеризация и выбор моделей выполняются по внутренней validation-части, holdout используется только для финальной оценки.

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import os, time, math, gc, ast
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120})

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, median_absolute_error,
    pairwise_distances, pairwise_distances_argmin_min
)
from sklearn.cluster import (
    MiniBatchKMeans, AffinityPropagation, MeanShift, estimate_bandwidth,
    SpectralClustering, AgglomerativeClustering, DBSCAN, OPTICS, Birch
)
from sklearn.mixture import GaussianMixture

try:
    import hdbscan
    HDBSCAN_AVAILABLE = True
except Exception:
    HDBSCAN_AVAILABLE = False

seed = 2
rng = np.random.RandomState(seed)
torch.manual_seed(seed)
np.random.seed(seed)

DATA_PATH_ENV = "DATA_FINALL_PATH"
ARTIFACT_DIR_ENV = "DL_CLUSTER_ARTIFACTS_DIR"


def require_path(env_name: str, label: str):
    value = os.environ.get(env_name)
    if not value:
        raise RuntimeError(f"Укажи путь к {label} в переменной окружения {env_name}.")
    path = Path(value).expanduser()
    if not path.exists():
        raise FileNotFoundError(f"Файл для {label} не найден: {path}")
    return path


DATA_PATH = require_path(DATA_PATH_ENV, "data_finall")
ARTIFACT_DIR = Path(os.environ.get(ARTIFACT_DIR_ENV, "./artifacts_dl_cluster_final_clean")).expanduser()
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)

target_col = "duration_hours"
DURATION_CAP = 960

# chronological split по порядку строк
split_idx = int(len(df) * 0.8)
train_raw = df.iloc[:split_idx].copy().reset_index(drop=True)
holdout_raw = df.iloc[split_idx:].copy().reset_index(drop=True)


train_typical = train_raw[train_raw[target_col] < DURATION_CAP].copy().reset_index(drop=True)
holdout_typical = holdout_raw[holdout_raw[target_col] < DURATION_CAP].copy().reset_index(drop=True)

Xtrain = train_typical.drop(columns=[target_col], errors="ignore").select_dtypes(include=["number"]).reset_index(drop=True)
ytrain = train_typical[target_col].reset_index(drop=True)

Xholdout_full = holdout_raw.drop(columns=[target_col], errors="ignore").select_dtypes(include=["number"]).reset_index(drop=True)
yholdout_full = holdout_raw[target_col].reset_index(drop=True)

Xholdout_typical = holdout_typical.drop(columns=[target_col], errors="ignore").select_dtypes(include=["number"]).reset_index(drop=True)
yholdout_typical = holdout_typical[target_col].reset_index(drop=True)

print(f"Train typical: {Xtrain.shape}, holdout full: {Xholdout_full.shape}, holdout typical: {Xholdout_typical.shape}")


val_cut = int(len(Xtrain) * 0.8)

screen_Xtr0 = Xtrain.iloc[:val_cut].copy().reset_index(drop=True)
screen_ytr0 = ytrain.iloc[:val_cut].values.astype(np.float32)

screen_Xva0 = Xtrain.iloc[val_cut:].copy().reset_index(drop=True)
screen_yva0 = ytrain.iloc[val_cut:].values.astype(np.float32)

# full typical train (for final refit / TabR store)
cluster_Xtr0 = Xtrain.copy().reset_index(drop=True)
cluster_ytr0 = ytrain.values.astype(np.float32)

cluster_Xte0 = Xholdout_full.copy().reset_index(drop=True)
cluster_yte0 = yholdout_full.values.astype(np.float32)

cluster_Xtyp0 = Xholdout_typical.copy().reset_index(drop=True)
cluster_ytyp0 = yholdout_typical.values.astype(np.float32)

# ---- Clustering preprocessing (train-only) ----
cluster_scaler = StandardScaler()
cluster_Xtr_scaled = cluster_scaler.fit_transform(cluster_Xtr0)
cluster_Xte_scaled = cluster_scaler.transform(cluster_Xte0)
cluster_Xtyp_scaled = cluster_scaler.transform(cluster_Xtyp0)

screen_Xtr_scaled = cluster_scaler.transform(screen_Xtr0)
screen_Xva_scaled = cluster_scaler.transform(screen_Xva0)

cluster_pca_n_components = min(30, cluster_Xtr0.shape[1])
cluster_pca = PCA(n_components=cluster_pca_n_components, random_state=seed)
cluster_Xtr_embed = cluster_pca.fit_transform(cluster_Xtr_scaled)
cluster_Xte_embed = cluster_pca.transform(cluster_Xte_scaled)
cluster_Xtyp_embed = cluster_pca.transform(cluster_Xtyp_scaled)

screen_Xtr_embed = cluster_pca.transform(screen_Xtr_scaled)
screen_Xva_embed = cluster_pca.transform(screen_Xva_scaled)

print(f"cluster_Xtr_embed(full train): {cluster_Xtr_embed.shape}")
print(f"screen_Xtr_embed: {screen_Xtr_embed.shape}, screen_Xva_embed: {screen_Xva_embed.shape}")
print(f"HDBSCAN available: {HDBSCAN_AVAILABLE}")

# ---- DL setup ----
device = torch.device(
    "mps" if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
INPUT_DIM = Xtrain.shape[1]
print(f"Устройство: {device}, признаков: {INPUT_DIM}")

# ---- Clustering infrastructure ----
HEAVY_CLUSTERERS = {
    'AffinityPropagation', 'MeanShift', 'SpectralClustering',
    'Ward', 'AgglomerativeClustering', 'DBSCAN', 'OPTICS', 'HDBSCAN'
}

CLUSTER_MAX_HEAVY_FIT_ROWS = 3000
CLUSTER_MIN_VALID_CLUSTERS = 2
CLUSTER_MAX_VALID_CLUSTERS = 20

def cluster_make_estimator(name, params):
    if name == 'MiniBatchKMeans':
        return MiniBatchKMeans(
            n_clusters=params['n_clusters'],
            random_state=seed,
            n_init=20,
            batch_size=1024
        )
    elif name == 'AffinityPropagation':
        # preference_quantile поддерживаем вручную ниже
        return AffinityPropagation(
            damping=params['damping'],
            random_state=seed
        )
    elif name == 'MeanShift':
        return MeanShift(
            bandwidth=params['bandwidth']
        )
    elif name == 'SpectralClustering':
        return SpectralClustering(
            n_clusters=params['n_clusters'],
            affinity=params['affinity'],
            random_state=seed,
            assign_labels='kmeans'
        )
    elif name == 'Ward':
        return AgglomerativeClustering(
            n_clusters=params['n_clusters'],
            linkage='ward'
        )
    elif name == 'AgglomerativeClustering':
        return AgglomerativeClustering(
            n_clusters=params['n_clusters'],
            linkage=params['linkage']
        )
    elif name == 'DBSCAN':
        return DBSCAN(
            eps=params['eps'],
            min_samples=params['min_samples']
        )
    elif name == 'OPTICS':
        return OPTICS(
            min_samples=params['min_samples'],
            xi=params['xi']
        )
    elif name == 'BIRCH':
        return Birch(
            n_clusters=params['n_clusters'],
            threshold=params['threshold']
        )
    elif name == 'GaussianMixture':
        return GaussianMixture(
            n_components=params['n_components'],
            covariance_type=params['covariance_type'],
            random_state=seed
        )
    elif name == 'HDBSCAN':
        return hdbscan.HDBSCAN(
            min_cluster_size=params['min_cluster_size'],
            min_samples=params['min_samples'],
            prediction_data=True
        )
    else:
        raise ValueError(name)

def cluster_build_centroids(X, labels):
    valid_mask = labels != -1
    valid_labels = np.unique(labels[valid_mask])

    if len(valid_labels) < CLUSTER_MIN_VALID_CLUSTERS:
        return None, None

    label_map = {old: new for new, old in enumerate(sorted(valid_labels))}
    mapped_labels = np.full_like(labels, fill_value=-1)

    centroids = []
    for old_lab in sorted(valid_labels):
        mask = labels == old_lab
        mapped_labels[mask] = label_map[old_lab]
        centroids.append(X[mask].mean(axis=0))

    centroids = np.vstack(centroids)

    if len(centroids) > CLUSTER_MAX_VALID_CLUSTERS:
        return None, None

    return centroids, mapped_labels

def cluster_assign_by_centroid(X, centroids):
    labels, dmin = pairwise_distances_argmin_min(X, centroids, metric='euclidean')
    all_d = pairwise_distances(X, centroids, metric='euclidean')
    return labels, dmin, all_d

def cluster_soft_probs_from_dist(all_d):
    inv_d = 1.0 / (all_d + 1e-6)
    return inv_d / inv_d.sum(axis=1, keepdims=True)

def cluster_fit_and_assign(clusterer_name, params, Xtr_embed, Xte_embed):
    Xfit = Xtr_embed
    fit_mode = "full_train"

    if clusterer_name in HEAVY_CLUSTERERS and len(Xtr_embed) > CLUSTER_MAX_HEAVY_FIT_ROWS:
        idx = rng.choice(len(Xtr_embed), size=CLUSTER_MAX_HEAVY_FIT_ROWS, replace=False)
        idx = np.sort(idx)
        Xfit = Xtr_embed[idx]
        fit_mode = f"subsample_{CLUSTER_MAX_HEAVY_FIT_ROWS}"

    est = cluster_make_estimator(clusterer_name, params)

    # GaussianMixture
    if clusterer_name == 'GaussianMixture':
        est.fit(Xfit)
        tr_labels = est.predict(Xtr_embed)
        te_labels = est.predict(Xte_embed)
        tr_proba = est.predict_proba(Xtr_embed)
        te_proba = est.predict_proba(Xte_embed)

        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    # HDBSCAN
    if clusterer_name == 'HDBSCAN':
        est.fit(Xfit)
        fit_labels = est.labels_

        valid = fit_labels != -1
        if len(np.unique(fit_labels[valid])) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        try:
            _te_labels_raw, _ = hdbscan.approximate_predict(est, Xte_embed)
        except Exception:
            return None

        centroids, _ = cluster_build_centroids(Xfit, fit_labels)
        if centroids is None:
            return None

        tr_labels, _, tr_dist = cluster_assign_by_centroid(Xtr_embed, centroids)
        te_labels, _, te_dist = cluster_assign_by_centroid(Xte_embed, centroids)
        tr_proba = cluster_soft_probs_from_dist(tr_dist)
        te_proba = cluster_soft_probs_from_dist(te_dist)

        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native_test"

    # General fit_predict
    if clusterer_name == 'AffinityPropagation':
        # optionally set preference from similarity quantile
        if 'preference_quantile' in params:
            sims = -pairwise_distances(Xfit, metric='euclidean')
            q = params['preference_quantile']
            pref = np.quantile(sims[np.triu_indices_from(sims, k=1)], q)
            est.preference = pref
        fit_labels = est.fit_predict(Xfit)
    elif hasattr(est, 'fit_predict'):
        fit_labels = est.fit_predict(Xfit)
    else:
        est.fit(Xfit)
        if hasattr(est, 'labels_'):
            fit_labels = est.labels_
        elif hasattr(est, 'predict'):
            fit_labels = est.predict(Xfit)
        else:
            return None

    # native predict for supported centroid clusterers
    if hasattr(est, 'predict') and clusterer_name in {'MiniBatchKMeans', 'BIRCH'} and len(Xfit) == len(Xtr_embed):
        tr_labels = est.predict(Xtr_embed)
        te_labels = est.predict(Xte_embed)

        if hasattr(est, 'cluster_centers_'):
            tr_d = pairwise_distances(Xtr_embed, est.cluster_centers_, metric='euclidean')
            te_d = pairwise_distances(Xte_embed, est.cluster_centers_, metric='euclidean')
            tr_proba = cluster_soft_probs_from_dist(tr_d)
            te_proba = cluster_soft_probs_from_dist(te_d)
        else:
            tr_proba = te_proba = None

        if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
            return None

        return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_native"

    # fallback centroid assignment
    centroids, _ = cluster_build_centroids(Xfit, fit_labels)
    if centroids is None:
        return None

    tr_labels, _, tr_d = cluster_assign_by_centroid(Xtr_embed, centroids)
    te_labels, _, te_d = cluster_assign_by_centroid(Xte_embed, centroids)
    tr_proba = cluster_soft_probs_from_dist(tr_d)
    te_proba = cluster_soft_probs_from_dist(te_d)

    if len(np.unique(tr_labels)) < CLUSTER_MIN_VALID_CLUSTERS:
        return None

    return tr_labels, te_labels, tr_proba, te_proba, fit_mode + "_centroid"

def cluster_build_meta_features(tr_labels, te_labels, tr_proba=None, te_proba=None):
    """
    ВАЖНО: без target leakage.
    НЕТ cluster_target_mean_train / cluster_target_median_train.
    """
    tr_feat = pd.DataFrame({'cluster_id': tr_labels})
    te_feat = pd.DataFrame({'cluster_id': te_labels})

    tr_sizes = pd.Series(tr_labels).value_counts().to_dict()
    tr_feat['cluster_size_train'] = pd.Series(tr_labels).map(tr_sizes).fillna(0)
    te_feat['cluster_size_train'] = pd.Series(te_labels).map(tr_sizes).fillna(0)

    tr_feat['is_noise'] = (tr_feat['cluster_id'] == -1).astype(int)
    te_feat['is_noise'] = (te_feat['cluster_id'] == -1).astype(int)

    tr_ohe = pd.get_dummies(tr_feat['cluster_id'], prefix='cluster')
    te_ohe = pd.get_dummies(te_feat['cluster_id'], prefix='cluster')
    te_ohe = te_ohe.reindex(columns=tr_ohe.columns, fill_value=0)

    tr_feat = pd.concat([tr_feat, tr_ohe], axis=1)
    te_feat = pd.concat([te_feat, te_ohe], axis=1)

    if tr_proba is not None and te_proba is not None:
        for k in range(tr_proba.shape[1]):
            tr_feat[f'cluster_proba_{k}'] = tr_proba[:, k]
            te_feat[f'cluster_proba_{k}'] = te_proba[:, k]

    return tr_feat, te_feat

# 11 clusterers as in current notebook
CLUSTER_EXPERIMENTS = {
    'MiniBatchKMeans': [
        {'n_clusters': k} for k in [2, 3, 4, 5, 6, 8]
    ],
    'AffinityPropagation': [
        {'damping': 0.7, 'preference_quantile': 0.1},
        {'damping': 0.8, 'preference_quantile': 0.2},
    ],
    'MeanShift': [
        {'bandwidth': 1.5},
        {'bandwidth': 2.0},
        {'bandwidth': 3.0},
    ],
    'SpectralClustering': [
        {'n_clusters': k, 'affinity': a}
        for k in [2, 3, 4, 5, 6]
        for a in ['nearest_neighbors', 'rbf']
    ],
    'Ward': [
        {'n_clusters': k} for k in [2, 3, 4, 5, 6, 8]
    ],
    'AgglomerativeClustering': [
        {'n_clusters': k, 'linkage': l}
        for k in [2, 3, 4, 5, 6, 8]
        for l in ['complete', 'average']
    ],
    'DBSCAN': [
        {'eps': eps, 'min_samples': ms}
        for eps in [0.3, 0.5, 0.8, 1.0]
        for ms in [5, 10, 20]
    ],
    'OPTICS': [
        {'min_samples': ms, 'xi': xi}
        for ms in [5, 10, 20]
        for xi in [0.03, 0.05, 0.1]
    ],
    'BIRCH': [
        {'n_clusters': k, 'threshold': th}
        for k in [2, 3, 4, 5, 6, 8]
        for th in [0.3, 0.5, 0.7]
    ],
    'GaussianMixture': [
        {'n_components': k, 'covariance_type': cov}
        for k in [2, 3, 4, 5, 6, 8]
        for cov in ['full', 'diag']
    ],
}
if HDBSCAN_AVAILABLE:
    CLUSTER_EXPERIMENTS['HDBSCAN'] = [
        {'min_cluster_size': mcs, 'min_samples': ms}
        for mcs in [10, 20, 50]
        for ms in [None, 5, 10]
    ]

cluster_total_cfgs = sum(len(v) for v in CLUSTER_EXPERIMENTS.values())
print(f"Кластеризация: {cluster_total_cfgs} конфигураций")


In [ ]:
# ── 2.1  MLP ─────────────────────────────────────────────────

class MLP(nn.Module):
    def __init__(self, d_in, hidden_dims, dropout=0.3):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h),
                       nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)


# ── 2.2  ResMLP ──────────────────────────────────────────────

class ResBlock(nn.Module):
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim), nn.BatchNorm1d(dim), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(dim, dim), nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()
    def forward(self, x):
        return self.act(x + self.block(x))

class ResMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU())
        self.blocks = nn.Sequential(*[ResBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(self.proj(x)))


# ── 2.3  SNN ─────────────────────────────────────────────────

class SNN(nn.Module):
    def __init__(self, d_in, hidden_dims=[256, 128], alpha_dropout=0.1):
        super().__init__()
        layers = []
        prev = d_in
        for h in hidden_dims:
            layers += [nn.Linear(prev, h), nn.SELU(), nn.AlphaDropout(alpha_dropout)]
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.net = nn.Sequential(*layers)
        self._init_weights()
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='linear')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
    def forward(self, x):
        return self.net(x)


# ── 2.4  GatedMLP ───────────────────────────────────────────

class GatedBlock(nn.Module):
    def __init__(self, d_in, d_out, dropout):
        super().__init__()
        self.linear = nn.Linear(d_in, d_out * 2)
        self.bn = nn.BatchNorm1d(d_out)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        h = self.linear(x)
        h1, h2 = h.chunk(2, dim=-1)
        return self.drop(self.bn(h1 * torch.sigmoid(h2)))

class GatedMLP(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        blocks = [GatedBlock(d_in, hidden, dropout)]
        for _ in range(n_blocks - 1):
            blocks.append(GatedBlock(hidden, hidden, dropout))
        self.blocks = nn.Sequential(*blocks)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        return self.head(self.blocks(x))


# ── 2.5  GANDALF ─────────────────────────────────────────────

class GANDALF(nn.Module):
    def __init__(self, d_in, hidden=256, n_blocks=3, dropout=0.3):
        super().__init__()
        self.gate_fc = nn.Linear(d_in, d_in)
        layers = [nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        for _ in range(n_blocks - 1):
            layers += [nn.Linear(hidden, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout)]
        self.dnn = nn.Sequential(*layers)
        self.head = nn.Linear(hidden, 1)
    def forward(self, x):
        gate = torch.sigmoid(self.gate_fc(x))
        return self.head(self.dnn(x * gate))


# ── 2.6  DAE-MLP ────────────────────────────────────────────

class DAEMLP(nn.Module):
    def __init__(self, d_in, hidden=256, latent=64, noise=0.3, dropout=0.3):
        super().__init__()
        self.noise = noise
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, latent), nn.BatchNorm1d(latent), nn.SiLU(),
        )
        self.decoder = nn.Sequential(nn.Linear(latent, hidden), nn.SiLU(), nn.Linear(hidden, d_in))
        self.reg_head = nn.Sequential(nn.Dropout(dropout), nn.Linear(latent, 32), nn.SiLU(), nn.Linear(32, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x_in = x * (torch.rand_like(x) > self.noise).float() if self.training and self.noise > 0 else x
        z = self.encoder(x_in)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(z), x)
        return self.reg_head(z)


# ── 2.7  CNN1D ───────────────────────────────────────────────

class CNN1D(nn.Module):
    def __init__(self, d_in, channels=[64, 128, 64], ks=5, dropout=0.3):
        super().__init__()
        layers = []
        in_ch = 1
        for ch in channels:
            layers += [nn.Conv1d(in_ch, ch, ks, padding=ks // 2),
                       nn.BatchNorm1d(ch), nn.SiLU(), nn.Dropout(dropout)]
            in_ch = ch
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Sequential(nn.Linear(channels[-1], 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        x = x.unsqueeze(1)
        x = self.pool(self.conv(x))
        return self.head(x.squeeze(-1))


# ── 2.8  BiGRU ───────────────────────────────────────────────

class BiGRU(nn.Module):
    def __init__(self, d_in, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.gru = nn.GRU(input_size=1, hidden_size=hidden, num_layers=n_layers,
                          batch_first=True, bidirectional=True,
                          dropout=dropout if n_layers > 1 else 0)
        self.head = nn.Sequential(nn.Linear(hidden * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        _, h = self.gru(x.unsqueeze(-1))
        return self.head(torch.cat([h[-2], h[-1]], dim=1))


# ── 2.9  FT-Transformer ─────────────────────────────────────

class FTTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.10  TabTransformer ────────────────────────────────────

class TabTransformer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.col_emb = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.Linear(d_in * d_model + d_in, mlp_hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(mlp_hidden, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1)) + self.col_emb
        out = self.transformer(tok)
        pooled = out.reshape(out.size(0), -1)
        return self.head(torch.cat([pooled, x], dim=1))


# ── 2.11  SAINT ──────────────────────────────────────────────

class SAINTBlock(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.sa_norm = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ia_norm = nn.LayerNorm(d_model)
        self.inter_attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.ffn = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, d_model * 4),
                                 nn.GELU(), nn.Dropout(dropout), nn.Linear(d_model * 4, d_model), nn.Dropout(dropout))
    def forward(self, x):
        h = self.sa_norm(x)
        h, _ = self.self_attn(h, h, h)
        x = x + h
        B, F_, D = x.shape
        if B > 1 and self.training:
            xt = x.permute(1, 0, 2)
            h2 = self.ia_norm(xt)
            h2, _ = self.inter_attn(h2, h2, h2)
            x = (xt + h2).permute(1, 0, 2)
        return x + self.ffn(x)

class SAINT(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        self.blocks = nn.ModuleList([SAINTBlock(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        tok = self.feat_emb(x.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        for blk in self.blocks:
            tok = blk(tok)
        return self.head(tok[:, 0])


# ── 2.12  AutoInt ────────────────────────────────────────────

class AutoIntLayer(nn.Module):
    def __init__(self, d_model, n_heads, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.W_res = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x):
        h, _ = self.attn(x, x, x)
        return F.relu(self.norm(h + self.W_res(x)))

class AutoInt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([AutoIntLayer(d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in * d_model, 256), nn.ReLU(), nn.Dropout(dropout), nn.Linear(256, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        for layer in self.layers:
            tok = layer(tok)
        return self.head(tok.reshape(tok.size(0), -1))


# ── 2.13  Trompt ─────────────────────────────────────────────

class TromptLayer(nn.Module):
    def __init__(self, d_in, d_model, n_heads, dropout):
        super().__init__()
        self.prompt = nn.Parameter(torch.randn(1, d_in, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(nn.Linear(d_model, d_model * 2), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(d_model * 2, d_model))
        self.norm2 = nn.LayerNorm(d_model)
        self.gate = nn.Linear(d_model, 1)
    def forward(self, feat_emb, x_raw):
        B = feat_emb.shape[0]
        prompted = feat_emb + self.prompt.expand(B, -1, -1)
        h, _ = self.attn(prompted, prompted, prompted)
        h = self.norm1(prompted + h)
        h = self.norm2(h + self.ffn(h))
        weights = torch.softmax(self.gate(h).squeeze(-1), dim=1)
        return h, weights * x_raw

class Trompt(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.layers = nn.ModuleList([TromptLayer(d_in, d_model, n_heads, dropout) for _ in range(n_layers)])
        self.head = nn.Sequential(nn.Linear(d_in, 128), nn.ReLU(), nn.Dropout(dropout), nn.Linear(128, 1))
    def forward(self, x):
        tok = self.feat_emb(x.unsqueeze(-1))
        preds = []
        for layer in self.layers:
            tok, lp = layer(tok, x)
            preds.append(lp)
        return self.head(torch.stack(preds).mean(0))


# ── 2.14  ExcelFormer ───────────────────────────────────────

class ExcelFormer(nn.Module):
    def __init__(self, d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2):
        super().__init__()
        self.feat_emb = nn.Linear(1, d_model)
        self.importance = nn.Parameter(torch.zeros(d_in))
        self.cls = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.pos = nn.Parameter(torch.randn(1, d_in + 1, d_model) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_heads,
              dim_feedforward=d_model * 4, dropout=dropout, activation="gelu", batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=n_layers)
        self.head = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))
    def forward(self, x):
        B = x.shape[0]
        mask = torch.sigmoid(self.importance)
        x_masked = x * mask
        tok = self.feat_emb(x_masked.unsqueeze(-1))
        tok = torch.cat([self.cls.expand(B, -1, -1), tok], dim=1) + self.pos
        return self.head(self.transformer(tok)[:, 0])


# ── 2.15  ARM-Net ────────────────────────────────────────────

class ARMNet(nn.Module):
    def __init__(self, d_in, n_exp=64, hidden=128, order=2, dropout=0.2):
        super().__init__()
        self.order = order
        self.exp_layers = nn.ModuleList([nn.Linear(d_in, n_exp) for _ in range(order)])
        self.gate = nn.Sequential(nn.Linear(d_in, n_exp * order), nn.Sigmoid())
        self.head = nn.Sequential(
            nn.BatchNorm1d(n_exp * order), nn.Linear(n_exp * order, hidden),
            nn.SiLU(), nn.Dropout(dropout), nn.Linear(hidden, 1))
    def forward(self, x):
        interactions = []
        for i, layer in enumerate(self.exp_layers):
            h = layer(x)
            if i > 0:
                h = h * interactions[-1]
            interactions.append(F.softplus(h))
        cat = torch.cat(interactions, dim=1)
        return self.head(cat * self.gate(x))


# ── 2.16  NODE ───────────────────────────────────────────────

class NODE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_weights = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.01)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        choices = torch.sigmoid(torch.einsum('bj,tdj->btd', x, self.feat_weights) - self.thresholds)
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        return self.drop(tree_out).mean(dim=-1, keepdim=True)


# ── 2.17  GRANDE ─────────────────────────────────────────────

class GRANDE(nn.Module):
    def __init__(self, d_in, n_trees=32, depth=4, dropout=0.0):
        super().__init__()
        self.n_trees = n_trees
        self.depth = depth
        n_leaves = 2 ** depth
        self.feat_logits = nn.Parameter(torch.randn(n_trees, depth, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_trees, depth))
        self.leaf_values = nn.Parameter(torch.randn(n_trees, n_leaves) * 0.01)
        self.tree_weights = nn.Parameter(torch.ones(n_trees) / n_trees)
        self.drop = nn.Dropout(dropout)
        patterns = torch.zeros(n_leaves, depth)
        for i in range(n_leaves):
            for d in range(depth):
                patterns[i, d] = (i >> (depth - 1 - d)) & 1
        self.register_buffer("patterns", patterns)
    def forward(self, x):
        feat_sel = F.softmax(self.feat_logits, dim=-1)
        proj = torch.einsum('bj,tdj->btd', x, feat_sel)
        choices = torch.sigmoid(10.0 * (proj - self.thresholds))
        c = choices.unsqueeze(2)
        p = self.patterns.unsqueeze(0).unsqueeze(0)
        leaf_probs = (c * p + (1 - c) * (1 - p)).prod(dim=-1)
        tree_out = (leaf_probs * self.leaf_values.unsqueeze(0)).sum(dim=-1)
        w = F.softmax(self.tree_weights, dim=0)
        return self.drop((tree_out * w).sum(dim=-1, keepdim=True))


# ── 2.18  Net-DNF ────────────────────────────────────────────

class NetDNF(nn.Module):
    def __init__(self, d_in, n_formulas=64, n_conjuncts=6, dropout=0.2):
        super().__init__()
        self.feat_sel = nn.Parameter(torch.randn(n_formulas, n_conjuncts, d_in) * 0.1)
        self.thresholds = nn.Parameter(torch.zeros(n_formulas, n_conjuncts))
        self.formula_w = nn.Linear(n_formulas, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, x):
        sel = torch.sigmoid(self.feat_sel)
        proj = torch.einsum('bj,fcj->bfc', x, sel)
        conjuncts = torch.sigmoid(proj - self.thresholds)
        formulas = conjuncts.prod(dim=-1)
        return self.formula_w(self.drop(formulas))


# ── 2.19  TabNet (simplified) ───────────────────────────────

class TabNet(nn.Module):
    def __init__(self, d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1):
        super().__init__()
        self.n_steps = n_steps
        self.relax = relax
        self.bn_init = nn.BatchNorm1d(d_in)
        self.shared_fc = nn.Linear(d_in, n_d + n_a)
        self.step_fcs = nn.ModuleList([nn.Linear(d_in, n_d + n_a) for _ in range(n_steps)])
        self.attn_fcs = nn.ModuleList([nn.Sequential(nn.Linear(n_a, d_in), nn.BatchNorm1d(d_in)) for _ in range(n_steps)])
        self.head = nn.Linear(n_d, 1)
        self.n_d = n_d
        self.drop = nn.Dropout(dropout)
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        x = self.bn_init(x)
        prior_scales = torch.ones(x.shape[0], x.shape[1], device=x.device)
        agg = torch.zeros(x.shape[0], self.n_d, device=x.device)
        entropy_loss = 0.0
        for i in range(self.n_steps):
            h = self.shared_fc(x * prior_scales) + self.step_fcs[i](x * prior_scales)
            h_d, h_a = h[:, :self.n_d], h[:, self.n_d:]
            h_d = F.silu(h_d)
            agg = agg + self.drop(h_d)
            attn = self.attn_fcs[i](h_a)
            attn = F.softmax(attn * prior_scales, dim=-1)
            prior_scales = prior_scales * (self.relax - attn)
            entropy_loss += (-attn * torch.log(attn + 1e-15)).mean()
        if self.training:
            self.aux_loss = entropy_loss / self.n_steps
        return self.head(agg)


# ── 2.20  TabR ───────────────────────────────────────────────

class TabR(nn.Module):
    def __init__(self, d_in, hidden=256, n_layers=3, k=32, dropout=0.3):
        super().__init__()
        self.k = k
        layers = []
        prev = d_in
        for i in range(n_layers):
            h = hidden if i == 0 else hidden // 2
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.SiLU(), nn.Dropout(dropout)]
            prev = h
        self.encoder = nn.Sequential(*layers)
        self.latent_dim = prev
        self.retrieval_head = nn.Sequential(nn.Linear(prev * 2, 128), nn.SiLU(), nn.Dropout(dropout), nn.Linear(128, 1))
        self.direct_head = nn.Linear(prev, 1)
        self._store_z = None
        self._store_y = None
    def build_store(self, x_train, y_train):
        self.eval()
        with torch.no_grad():
            self._store_z = self.encoder(x_train)
            self._store_y = y_train
    def forward(self, x):
        z = self.encoder(x)
        if self._store_z is not None and not self.training:
            dists = torch.cdist(z, self._store_z)
            _, idx = dists.topk(self.k, largest=False)
            nbr_z = self._store_z[idx]
            sim = -dists.gather(1, idx)
            attn = torch.softmax(sim, dim=1).unsqueeze(-1)
            context = (attn * nbr_z).sum(dim=1)
            return self.retrieval_head(torch.cat([z, context], dim=1))
        return self.direct_head(z)


# ── 2.21  GrowNet stages ────────────────────────────────────

class GrowNetStage(nn.Module):
    def __init__(self, d_in, hidden=32, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in + 1, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden), nn.SiLU(), nn.Linear(hidden, 1))
    def forward(self, x, prev_pred):
        return self.net(torch.cat([x, prev_pred], dim=1))


# ── 2.22  SwitchTab ──────────────────────────────────────────

class SwitchTab(nn.Module):
    def __init__(self, d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(d_in, d_enc), nn.BatchNorm1d(d_enc), nn.SiLU(), nn.Dropout(dropout))
        self.mutual_proj = nn.Linear(d_enc, d_mutual)
        self.salient_proj = nn.Linear(d_enc, d_salient)
        self.decoder = nn.Sequential(nn.Linear(d_mutual + d_salient, d_in))
        self.head = nn.Sequential(nn.Linear(d_mutual + d_salient, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
        self.aux_loss = torch.tensor(0.0)
    def forward(self, x):
        h = self.encoder(x)
        mutual = self.mutual_proj(h)
        salient = self.salient_proj(h)
        rep = torch.cat([mutual, salient], dim=1)
        if self.training:
            self.aux_loss = F.mse_loss(self.decoder(rep), x)
        return self.head(rep)


# ── 2.23  PTaRL ──────────────────────────────────────────────

class PTaRL(nn.Module):
    def __init__(self, d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, d_latent), nn.BatchNorm1d(d_latent), nn.SiLU())
        self.prototypes = nn.Parameter(torch.randn(n_prototypes, d_latent) * 0.1)
        self.head = nn.Sequential(nn.Linear(d_latent * 2, 64), nn.SiLU(), nn.Dropout(dropout), nn.Linear(64, 1))
    def forward(self, x):
        z = self.encoder(x)
        sim = F.cosine_similarity(z.unsqueeze(1), self.prototypes.unsqueeze(0), dim=-1)
        attn = F.softmax(sim * 10, dim=1)
        proto_rep = (attn.unsqueeze(-1) * self.prototypes.unsqueeze(0)).sum(dim=1)
        return self.head(torch.cat([z, proto_rep], dim=1))


# ── 2.24  DCN v2 ────────────────────────────────────────────

class CrossLayer(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.W = nn.Linear(d, d, bias=False)
        self.b = nn.Parameter(torch.zeros(d))
    def forward(self, x0, x):
        return x0 * self.W(x) + self.b + x

class DCNv2(nn.Module):
    def __init__(self, d_in, n_cross=3, hidden=256, dropout=0.3):
        super().__init__()
        self.cross_layers = nn.ModuleList([CrossLayer(d_in) for _ in range(n_cross)])
        self.deep = nn.Sequential(
            nn.Linear(d_in, hidden), nn.BatchNorm1d(hidden), nn.SiLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.BatchNorm1d(hidden // 2), nn.SiLU(), nn.Dropout(dropout))
        self.head = nn.Linear(d_in + hidden // 2, 1)
    def forward(self, x):
        x_cross = x
        for cl in self.cross_layers:
            x_cross = cl(x, x_cross)
        return self.head(torch.cat([x_cross, self.deep(x)], dim=1))


# ═══════════════════════════════════════════════════════════════
#  Baseline MAE для каждой DL-модели (из предыдущего прогона)
#  Нужен для расчёта delta = baseline - cluster_mae
# ═══════════════════════════════════════════════════════════════

dl_baseline_mae = {
    "MLP":            246.89,
    "ResMLP":         245.32,
    "SNN":            252.41,
    "GatedMLP":       247.15,
    "GANDALF":        246.53,
    "DAE-MLP":        253.78,
    "CNN1D":          251.64,
    "BiGRU":          254.92,
    "FT-Transformer": 248.37,
    "TabTransformer": 249.81,
    "SAINT":          250.14,
    "AutoInt":        248.96,
    "Trompt":         251.23,
    "ExcelFormer":    249.45,
    "ARM-Net":        250.67,
    "NODE":           247.82,
    "GRANDE":         248.11,
    "Net-DNF":        252.03,
    "TabNet":         249.58,
    "TabR":           246.41,
    "GrowNet":        247.29,
    "SwitchTab":      251.87,
    "PTaRL":          250.93,
    "DCNv2":          247.64,
}


# ═══════════════════════════════════════════════════════════════
#  Функции обучения на аугментированных данных
# ═══════════════════════════════════════════════════════════════

def train_model_aug(model, X_tr, y_tr, X_va, y_va,
                    epochs=300, lr=1e-3, wd=1e-4, bs=256,
                    patience=30, aux_w=0.0):
    """Обучение с early stopping на произвольных данных (не глобальных)."""
    model = model.to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
    loss_fn = nn.L1Loss()
    ds = TensorDataset(X_tr, y_tr)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    X_v, y_v = X_va.to(device), y_va.to(device)

    best_val, best_state, wait = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            pred = model(xb)
            loss = loss_fn(pred, yb)
            if aux_w > 0 and hasattr(model, "aux_loss"):
                loss = loss + aux_w * model.aux_loss
            opt.zero_grad(); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad():
            v_mae = loss_fn(model(X_v), y_v).item()
        sched.step(v_mae)
        if v_mae < best_val:
            best_val = v_mae
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    model.load_state_dict(best_state)
    return model, best_val, ep


def train_grownet_aug(X_tr, y_tr, X_va, y_va, X_te,
                      n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                      step_size=0.1, bs=256, patience=30, dropout=0.1):
    """GrowNet на произвольных данных."""
    d_in = X_tr.shape[1]
    stages = []
    X_v, y_v = X_va.to(device), y_va.to(device)
    ds = TensorDataset(X_tr, y_tr)
    dl = DataLoader(ds, batch_size=bs, shuffle=True)
    loss_fn = nn.L1Loss()

    def ensemble_pred(x, stgs):
        p = torch.zeros(x.size(0), 1, device=x.device)
        for s in stgs:
            s.eval()
            p = p + step_size * s(x, p.detach())
        return p

    best_val, best_n = float("inf"), 0
    for stage_idx in range(n_stages):
        model = GrowNetStage(d_in, hidden, dropout).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
        sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", patience=10, factor=0.5, min_lr=1e-6)
        best_stage_val, best_stage_state, wait = float("inf"), None, 0
        for ep in range(1, 201):
            model.train()
            for xb, yb in dl:
                xb, yb = xb.to(device), yb.to(device)
                with torch.no_grad():
                    prev = ensemble_pred(xb, stages)
                correction = model(xb, prev)
                loss = loss_fn(prev + step_size * correction, yb)
                opt.zero_grad(); loss.backward(); opt.step()
            model.eval()
            with torch.no_grad():
                prev_v = ensemble_pred(X_v, stages)
                v_pred = prev_v + step_size * model(X_v, prev_v)
                v_mae = loss_fn(v_pred, y_v).item()
            sched.step(v_mae)
            if v_mae < best_stage_val:
                best_stage_val = v_mae
                best_stage_state = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1
                if wait >= patience:
                    break
        model.load_state_dict(best_stage_state)
        stages.append(model)
        with torch.no_grad():
            ens_val = loss_fn(ensemble_pred(X_v, stages), y_v).item()
        if ens_val < best_val:
            best_val = ens_val
            best_n = len(stages)
    stages = stages[:best_n]
    return stages, best_val


def make_dl_builders(d_in):
    """Фабрики всех 24 архитектур для заданной размерности."""
    return {
        "MLP":            lambda: MLP(d_in, [256, 128], dropout=0.3),
        "ResMLP":         lambda: ResMLP(d_in, hidden=256, n_blocks=3, dropout=0.3),
        "SNN":            lambda: SNN(d_in, hidden_dims=[256, 128], alpha_dropout=0.1),
        "GatedMLP":       lambda: GatedMLP(d_in, hidden=256, n_blocks=3, dropout=0.3),
        "GANDALF":        lambda: GANDALF(d_in, hidden=256, n_blocks=3, dropout=0.3),
        "DAE-MLP":        lambda: DAEMLP(d_in, hidden=256, latent=64, noise=0.3, dropout=0.3),
        "CNN1D":          lambda: CNN1D(d_in, channels=[64, 128, 64], ks=5, dropout=0.3),
        "BiGRU":          lambda: BiGRU(d_in, hidden=64, n_layers=2, dropout=0.3),
        "FT-Transformer": lambda: FTTransformer(d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
        "TabTransformer": lambda: TabTransformer(d_in, d_model=32, n_heads=4, n_layers=2, mlp_hidden=128, dropout=0.2),
        "SAINT":          lambda: SAINT(d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
        "AutoInt":        lambda: AutoInt(d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
        "Trompt":         lambda: Trompt(d_in, d_model=32, n_heads=4, n_layers=3, dropout=0.2),
        "ExcelFormer":    lambda: ExcelFormer(d_in, d_model=32, n_heads=4, n_layers=2, dropout=0.2),
        "ARM-Net":        lambda: ARMNet(d_in, n_exp=64, hidden=128, order=2, dropout=0.2),
        "NODE":           lambda: NODE(d_in, n_trees=32, depth=4, dropout=0.0),
        "GRANDE":         lambda: GRANDE(d_in, n_trees=32, depth=4, dropout=0.0),
        "Net-DNF":        lambda: NetDNF(d_in, n_formulas=64, n_conjuncts=6, dropout=0.2),
        "TabNet":         lambda: TabNet(d_in, n_steps=3, n_d=64, n_a=64, relax=1.5, dropout=0.1),
        "TabR":           lambda: TabR(d_in, hidden=256, n_layers=3, k=32, dropout=0.3),
        "SwitchTab":      lambda: SwitchTab(d_in, d_enc=128, d_mutual=64, d_salient=64, dropout=0.3),
        "PTaRL":          lambda: PTaRL(d_in, n_prototypes=16, d_latent=128, hidden=256, dropout=0.3),
        "DCNv2":          lambda: DCNv2(d_in, n_cross=3, hidden=256, dropout=0.3),
    }

DL_AUX_W = {"DAE-MLP": 0.1, "TabNet": 0.01, "SwitchTab": 0.1}

print("DL-архитектуры и функции обучения определены.")

In [ ]:

X_dl_tr = screen_Xtr0.values.astype(np.float32)
y_dl_tr = screen_ytr0.astype(np.float32)

X_dl_va = screen_Xva0.values.astype(np.float32)
y_dl_va = screen_yva0.astype(np.float32)

X_dl_te_full = cluster_Xte0.values.astype(np.float32)
X_dl_te_typ = cluster_Xtyp0.values.astype(np.float32)

sc_dl = StandardScaler()
X_dl_tr = sc_dl.fit_transform(X_dl_tr)
X_dl_va = sc_dl.transform(X_dl_va)
X_dl_te_full = sc_dl.transform(X_dl_te_full)
X_dl_te_typ = sc_dl.transform(X_dl_te_typ)

for arr in [X_dl_tr, X_dl_va, X_dl_te_full, X_dl_te_typ]:
    np.nan_to_num(arr, copy=False)

X_dl_tr_t = torch.from_numpy(X_dl_tr)
y_dl_tr_t = torch.from_numpy(y_dl_tr).unsqueeze(1)
X_dl_va_t = torch.from_numpy(X_dl_va)
y_dl_va_t = torch.from_numpy(y_dl_va).unsqueeze(1)
X_dl_te_full_t = torch.from_numpy(X_dl_te_full)
X_dl_te_typ_t = torch.from_numpy(X_dl_te_typ)

# full typical train for TabR store / final consistency
X_dl_full = cluster_Xtr0.values.astype(np.float32)
sc_dl_full = StandardScaler()
X_dl_full = sc_dl_full.fit_transform(X_dl_full)
np.nan_to_num(X_dl_full, copy=False)
X_dl_full_t = torch.from_numpy(X_dl_full)
y_dl_full_t = torch.from_numpy(cluster_ytr0.astype(np.float32)).unsqueeze(1)

dl_baseline_val_mae = {}
dl_baseline_holdout_full_mae = {}
dl_baseline_holdout_typ_mae = {}
dl_baseline_rows = []

all_builders = make_dl_builders(INPUT_DIM)

print("="*85)
print("DL baselines")
print("="*85)

for dl_name, build_fn in all_builders.items():
    torch.manual_seed(seed)
    aux = DL_AUX_W.get(dl_name, 0.0)

    try:
        model = build_fn()
        model, val_mae, ep_used = train_model_aug(
            model, X_dl_tr_t, y_dl_tr_t,
            X_dl_va_t, y_dl_va_t,
            epochs=300, lr=1e-3, wd=1e-4, bs=256,
            patience=30, aux_w=aux
        )

        if dl_name == "TabR":
            model.build_store(X_dl_full_t.to(device), y_dl_full_t.to(device))

        model.eval()
        with torch.no_grad():
            pred_full = model(X_dl_te_full_t.to(device)).cpu().numpy().flatten()
            pred_typ = model(X_dl_te_typ_t.to(device)).cpu().numpy().flatten()

        mae_full = mean_absolute_error(cluster_yte0, pred_full)
        rmse_full = np.sqrt(mean_squared_error(cluster_yte0, pred_full))
        mdae_full = median_absolute_error(cluster_yte0, pred_full)

        mae_typ = mean_absolute_error(cluster_ytyp0, pred_typ)
        rmse_typ = np.sqrt(mean_squared_error(cluster_ytyp0, pred_typ))
        mdae_typ = median_absolute_error(cluster_ytyp0, pred_typ)

        dl_baseline_val_mae[dl_name] = val_mae
        dl_baseline_holdout_full_mae[dl_name] = mae_full
        dl_baseline_holdout_typ_mae[dl_name] = mae_typ

        dl_baseline_rows.append({
            "model": dl_name,
            "val_MAE": round(val_mae, 2),
            "holdout_typical_MAE": round(mae_typ, 2),
            "holdout_typical_RMSE": round(rmse_typ, 2),
            "holdout_typical_MdAE": round(mdae_typ, 2),
            "holdout_full_MAE": round(mae_full, 2),
            "holdout_full_RMSE": round(rmse_full, 2),
            "holdout_full_MdAE": round(mdae_full, 2),
            "epochs_used": int(ep_used),
        })
        print(f"{dl_name:<16s} val={val_mae:.2f} holdout_typ={mae_typ:.2f} holdout_full={mae_full:.2f}")
    except Exception as e:
        print(f"{dl_name:<16s} ошибка: {e}")

# GrowNet separately
try:
    torch.manual_seed(seed)
    gn_stages, gn_val = train_grownet_aug(
        X_dl_tr_t, y_dl_tr_t, X_dl_va_t, y_dl_va_t, X_dl_te_full_t,
        n_stages=5, hidden=32, lr=0.01, wd=1e-4,
        step_size=0.1, bs=256, patience=30
    )

    with torch.no_grad():
        p_full = torch.zeros(X_dl_te_full_t.size(0), 1, device=device)
        for s in gn_stages:
            s.eval()
            p_full = p_full + 0.1 * s(X_dl_te_full_t.to(device), p_full)
        gn_pred_full = p_full.cpu().numpy().flatten()

        p_typ = torch.zeros(X_dl_te_typ_t.size(0), 1, device=device)
        for s in gn_stages:
            s.eval()
            p_typ = p_typ + 0.1 * s(X_dl_te_typ_t.to(device), p_typ)
        gn_pred_typ = p_typ.cpu().numpy().flatten()

    gn_mae_full = mean_absolute_error(cluster_yte0, gn_pred_full)
    gn_rmse_full = np.sqrt(mean_squared_error(cluster_yte0, gn_pred_full))
    gn_mdae_full = median_absolute_error(cluster_yte0, gn_pred_full)

    gn_mae_typ = mean_absolute_error(cluster_ytyp0, gn_pred_typ)
    gn_rmse_typ = np.sqrt(mean_squared_error(cluster_ytyp0, gn_pred_typ))
    gn_mdae_typ = median_absolute_error(cluster_ytyp0, gn_pred_typ)

    dl_baseline_val_mae["GrowNet"] = gn_val
    dl_baseline_holdout_full_mae["GrowNet"] = gn_mae_full
    dl_baseline_holdout_typ_mae["GrowNet"] = gn_mae_typ

    dl_baseline_rows.append({
        "model": "GrowNet",
        "val_MAE": round(gn_val, 2),
        "holdout_typical_MAE": round(gn_mae_typ, 2),
        "holdout_typical_RMSE": round(gn_rmse_typ, 2),
        "holdout_typical_MdAE": round(gn_mdae_typ, 2),
        "holdout_full_MAE": round(gn_mae_full, 2),
        "holdout_full_RMSE": round(gn_rmse_full, 2),
        "holdout_full_MdAE": round(gn_mdae_full, 2),
        "epochs_used": None,
    })
    print(f"{'GrowNet':<16s} val={gn_val:.2f} holdout_typ={gn_mae_typ:.2f} holdout_full={gn_mae_full:.2f}")
except Exception as e:
    print(f"GrowNet ошибка: {e}")

dl_baseline_df = pd.DataFrame(dl_baseline_rows).sort_values("holdout_full_MAE").reset_index(drop=True)
display(dl_baseline_df.head(30))


In [ ]:

DL_BATCHES = {
    1: ["MLP", "ResMLP", "SNN", "GatedMLP"],
    2: ["GANDALF", "DAE-MLP", "CNN1D", "BiGRU"],
    3: ["FT-Transformer", "TabTransformer", "SAINT", "AutoInt"],
    4: ["Trompt", "ExcelFormer", "ARM-Net", "NODE"],
    5: ["GRANDE", "Net-DNF", "TabNet", "TabR"],
    6: ["SwitchTab", "PTaRL", "DCNv2", "GrowNet"],
}

dl_cluster_all_results = []

def run_one_clusterer_dl(clusterer_name, param_grid, dl_names=None, include_grownet=False):
    dl_cluster_rows = []
    total = len(param_grid)

    print(f'\n{"═" * 75}')
    print(f'  DL × {clusterer_name}: {total} конфигураций | DL: {dl_names}')
    print(f'{"═" * 75}')

    for i, params in enumerate(param_grid, 1):
        t0 = time.time()

        result = cluster_fit_and_assign(
            clusterer_name, params,
            screen_Xtr_embed, screen_Xva_embed
        )

        if result is None:
            print(f'  [{i:>2d}/{total}] {clusterer_name} {params} -> skip')
            continue

        tr_labels, va_labels, tr_proba, va_proba, fit_mode = result

        tr_feat, va_feat = cluster_build_meta_features(
            tr_labels, va_labels,
            tr_proba=tr_proba, te_proba=va_proba
        )

        cluster_Xtr_aug = pd.concat([screen_Xtr0.reset_index(drop=True), tr_feat.reset_index(drop=True)], axis=1)
        cluster_Xva_aug = pd.concat([screen_Xva0.reset_index(drop=True), va_feat.reset_index(drop=True)], axis=1)

        aug_d_in = cluster_Xtr_aug.shape[1]

        X_aug_tr_np = cluster_Xtr_aug.values.astype(np.float32)
        y_aug_tr_np = screen_ytr0.astype(np.float32)
        X_aug_va_np = cluster_Xva_aug.values.astype(np.float32)
        y_aug_va_np = screen_yva0.astype(np.float32)

        aug_sc = StandardScaler()
        X_aug_tr_np = aug_sc.fit_transform(X_aug_tr_np)
        X_aug_va_np = aug_sc.transform(X_aug_va_np)

        for arr in [X_aug_tr_np, X_aug_va_np]:
            np.nan_to_num(arr, copy=False)

        X_aug_tr_t = torch.from_numpy(X_aug_tr_np)
        y_aug_tr_t = torch.from_numpy(y_aug_tr_np).unsqueeze(1)
        X_aug_va_t = torch.from_numpy(X_aug_va_np)
        y_aug_va_t = torch.from_numpy(y_aug_va_np).unsqueeze(1)

        # full typical train for TabR store / final refit later
        full_aug_result = cluster_fit_and_assign(clusterer_name, params, cluster_Xtr_embed, cluster_Xtr_embed)
        if full_aug_result is None:
            print(f'  [{i:>2d}/{total}] {params} -> skip(full)')
            continue
        full_tr_labels, _, full_tr_proba, _, _ = full_aug_result
        full_tr_feat, _ = cluster_build_meta_features(full_tr_labels, full_tr_labels, tr_proba=full_tr_proba, te_proba=full_tr_proba)
        cluster_Xfull_aug = pd.concat([cluster_Xtr0.reset_index(drop=True), full_tr_feat.reset_index(drop=True)], axis=1)

        aug_sc_full = StandardScaler()
        X_aug_full_np = aug_sc_full.fit_transform(cluster_Xfull_aug.values.astype(np.float32))
        np.nan_to_num(X_aug_full_np, copy=False)
        X_aug_full_t = torch.from_numpy(X_aug_full_np)
        y_aug_full_t = torch.from_numpy(cluster_ytr0.astype(np.float32)).unsqueeze(1)

        all_builders = make_dl_builders(aug_d_in)
        builders = {k: v for k, v in all_builders.items() if dl_names is None or k in dl_names}
        deltas = []

        for dl_name, build_fn in builders.items():
            torch.manual_seed(seed)
            aux = DL_AUX_W.get(dl_name, 0.0)

            try:
                model = build_fn()
                model, val_mae, ep_used = train_model_aug(
                    model, X_aug_tr_t, y_aug_tr_t,
                    X_aug_va_t, y_aug_va_t,
                    epochs=300, lr=1e-3, wd=1e-4, bs=256,
                    patience=30, aux_w=aux
                )

                if dl_name == "TabR":
                    model.build_store(X_aug_full_t.to(device), y_aug_full_t.to(device))
            except Exception as e:
                print(f'    {dl_name}: ошибка — {e}')
                continue

            baseline_val = dl_baseline_val_mae.get(dl_name, val_mae)
            delta_val = baseline_val - val_mae

            deltas.append(delta_val)
            dl_cluster_rows.append(dict(
                clusterer=clusterer_name,
                params=str(params),
                fit_mode=fit_mode,
                model=dl_name,
                val_mae=round(val_mae, 2),
                baseline_val=round(baseline_val, 2),
                delta_val=round(delta_val, 2)
            ))

            gc.collect()
            if device.type == "mps":
                torch.mps.empty_cache()

        if include_grownet:
            torch.manual_seed(seed)
            try:
                gn_stages, gn_val = train_grownet_aug(
                    X_aug_tr_t, y_aug_tr_t,
                    X_aug_va_t, y_aug_va_t,
                    X_aug_va_t,
                    n_stages=5, hidden=32, lr=0.01, wd=1e-4,
                    step_size=0.1, bs=256, patience=30
                )
                gn_baseline_val = dl_baseline_val_mae.get("GrowNet", gn_val)
                gn_delta_val = gn_baseline_val - gn_val

                deltas.append(gn_delta_val)
                dl_cluster_rows.append(dict(
                    clusterer=clusterer_name,
                    params=str(params),
                    fit_mode=fit_mode,
                    model="GrowNet",
                    val_mae=round(gn_val, 2),
                    baseline_val=round(gn_baseline_val, 2),
                    delta_val=round(gn_delta_val, 2)
                ))
            except Exception as e:
                print(f'    GrowNet: ошибка — {e}')

        gc.collect()
        if device.type == "mps":
            torch.mps.empty_cache()

        avg_delta = np.mean(deltas) if deltas else 0
        best_delta = np.max(deltas) if deltas else 0
        elapsed = time.time() - t0
        print(f'  [{i:>2d}/{total}] {params}  avg Δval={avg_delta:+.2f}  best Δval={best_delta:+.2f}  ({elapsed:.1f}s)')

    return pd.DataFrame(dl_cluster_rows)

print("run_one_clusterer_dl defined.")
print(f"DL_BATCHES: {list(DL_BATCHES.values())}")


In [ ]:
# ── БАТЧ 1/6: MLP, ResMLP, SNN, GatedMLP ─────────────────────
_batch_names = DL_BATCHES[1]
print(f'{"═"*80}\n  БАТЧ 1/6: {_batch_names}\n{"═"*80}')
_t0 = time.time()
_batch1_results = []
for _cname, _pgrid in CLUSTER_EXPERIMENTS.items():
    _df = run_one_clusterer_dl(_cname, _pgrid, dl_names=_batch_names, include_grownet=False)
    _batch1_results.append(_df)
dl_cluster_all_results.extend(_batch1_results)
print(f'\nБатч 1 завершён за {time.time()-_t0:.0f}s, строк: {sum(len(d) for d in _batch1_results)}')

# ── Промежуточный анализ после батча 1 ──────────────────────
_interim_df = pd.concat(dl_cluster_all_results, ignore_index=True)
print(f'\n{"═" * 80}')
print(f'  ПРОМЕЖУТОЧНЫЙ АНАЛИЗ ПОСЛЕ БАТЧА 1/6 (накоплено {len(_interim_df)} строк)')
print(f'{"═" * 80}')

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО КЛАСТЕРИЗАТОРАМ:')
_interim_clust = _interim_df.groupby('clusterer')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_clust)

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО DL-МОДЕЛЯМ:')
_interim_model = _interim_df.groupby('model')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_model)

# Heatmap
_interim_pivot = _interim_df.pivot_table(
    index='clusterer',
    columns='model',
    values='delta_val',
    aggfunc='max'
)
_interim_pivot['avg'] = _interim_pivot.mean(axis=1)
_interim_pivot = _interim_pivot.sort_values('avg', ascending=False)
_interim_plot = _interim_pivot.drop(columns=['avg'])

fig, ax = plt.subplots(figsize=(max(12, len(_interim_plot.columns) * 1.2), 8))
sns.heatmap(
    _interim_plot,
    annot=True,
    fmt='.1f',
    cmap='RdYlGn',
    center=0,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Промежуточная heatmap после батча 1/6: Max Δ val MAE (>0 = улучшение)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

# Top-10
_interim_top = _interim_df.sort_values('delta_val', ascending=False).head(10).reset_index(drop=True)
print(f'\n  ТОП-10 ЛУЧШИХ КОМБИНАЦИЙ (на данный момент):')
display(_interim_top[['clusterer', 'params', 'fit_mode', 'model', 'val_mae', 'baseline_val', 'delta_val']])

_best = _interim_df.loc[_interim_df['delta_val'].idxmax()]
print(
    f'\nТекущий лидер: {_best["clusterer"]} + {_best["model"]} '
    f'— val_MAE={_best["val_mae"]:.2f} (Δval={_best["delta_val"]:+.2f})'
)
print(f'{"─" * 80}')

del _interim_df, _interim_clust, _interim_model, _interim_pivot, _interim_plot, _interim_top, _best
gc.collect()


In [ ]:
# ── БАТЧ 2/6: GANDALF, DAE-MLP, CNN1D, BiGRU ─────────────────
_batch_names = DL_BATCHES[2]
print(f'{"═"*80}\n  БАТЧ 2/6: {_batch_names}\n{"═"*80}')
_t0 = time.time()
_batch2_results = []
for _cname, _pgrid in CLUSTER_EXPERIMENTS.items():
    _df = run_one_clusterer_dl(_cname, _pgrid, dl_names=_batch_names, include_grownet=False)
    _batch2_results.append(_df)
dl_cluster_all_results.extend(_batch2_results)
print(f'\nБатч 2 завершён за {time.time()-_t0:.0f}s, строк: {sum(len(d) for d in _batch2_results)}')

# ── Промежуточный анализ после батча 2 ──────────────────────
_interim_df = pd.concat(dl_cluster_all_results, ignore_index=True)
print(f'\n{"═" * 80}')
print(f'  ПРОМЕЖУТОЧНЫЙ АНАЛИЗ ПОСЛЕ БАТЧА 2/6 (накоплено {len(_interim_df)} строк)')
print(f'{"═" * 80}')

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО КЛАСТЕРИЗАТОРАМ:')
_interim_clust = _interim_df.groupby('clusterer')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_clust)

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО DL-МОДЕЛЯМ:')
_interim_model = _interim_df.groupby('model')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_model)

_interim_pivot = _interim_df.pivot_table(index='clusterer', columns='model', values='delta_val', aggfunc='max')
_interim_pivot['avg'] = _interim_pivot.mean(axis=1)
_interim_pivot = _interim_pivot.sort_values('avg', ascending=False)
_interim_plot = _interim_pivot.drop(columns=['avg'])

fig, ax = plt.subplots(figsize=(max(12, len(_interim_plot.columns)*1.2), 8))
sns.heatmap(_interim_plot, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Промежуточная heatmap после батча 2/6: Max Δ val MAE (>0 = улучшение)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

_interim_top = _interim_df.sort_values('delta_val', ascending=False).head(10).reset_index(drop=True)
print(f'\n  ТОП-10 ЛУЧШИХ КОМБИНАЦИЙ (на данный момент):')
display(_interim_top[['clusterer', 'params', 'fit_mode', 'model', 'val_mae', 'baseline_val', 'delta_val']])

_best = _interim_df.loc[_interim_df['delta_val'].idxmax()]
print(
    f'\nТекущий лидер: {_best["clusterer"]} + {_best["model"]} '
    f'— val_MAE={_best["val_mae"]:.2f} (Δval={_best["delta_val"]:+.2f})'
)
print(f'{"─" * 80}')

del _interim_df, _interim_clust, _interim_model, _interim_pivot, _interim_plot, _interim_top, _best
gc.collect()


In [ ]:
# ── БАТЧ 3/6: FT-Transformer, TabTransformer, SAINT, AutoInt ──
_batch_names = DL_BATCHES[3]
print(f'{"═"*80}\n  БАТЧ 3/6: {_batch_names}\n{"═"*80}')
_t0 = time.time()
_batch3_results = []
for _cname, _pgrid in CLUSTER_EXPERIMENTS.items():
    _df = run_one_clusterer_dl(_cname, _pgrid, dl_names=_batch_names, include_grownet=False)
    _batch3_results.append(_df)
dl_cluster_all_results.extend(_batch3_results)
print(f'\nБатч 3 завершён за {time.time()-_t0:.0f}s, строк: {sum(len(d) for d in _batch3_results)}')

# ── Промежуточный анализ после батча 3 ──────────────────────
_interim_df = pd.concat(dl_cluster_all_results, ignore_index=True)
print(f'\n{"═" * 80}')
print(f'  ПРОМЕЖУТОЧНЫЙ АНАЛИЗ ПОСЛЕ БАТЧА 3/6 (накоплено {len(_interim_df)} строк)')
print(f'{"═" * 80}')

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО КЛАСТЕРИЗАТОРАМ:')
_interim_clust = _interim_df.groupby('clusterer')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_clust)

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО DL-МОДЕЛЯМ:')
_interim_model = _interim_df.groupby('model')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_model)

_interim_pivot = _interim_df.pivot_table(index='clusterer', columns='model', values='delta_val', aggfunc='max')
_interim_pivot['avg'] = _interim_pivot.mean(axis=1)
_interim_pivot = _interim_pivot.sort_values('avg', ascending=False)
_interim_plot = _interim_pivot.drop(columns=['avg'])

fig, ax = plt.subplots(figsize=(max(12, len(_interim_plot.columns)*1.2), 8))
sns.heatmap(_interim_plot, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Промежуточная heatmap после батча 3/6: Max Δ val MAE (>0 = улучшение)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

_interim_top = _interim_df.sort_values('delta_val', ascending=False).head(10).reset_index(drop=True)
print(f'\n  ТОП-10 ЛУЧШИХ КОМБИНАЦИЙ (на данный момент):')
display(_interim_top[['clusterer', 'params', 'fit_mode', 'model', 'val_mae', 'baseline_val', 'delta_val']])

_best = _interim_df.loc[_interim_df['delta_val'].idxmax()]
print(
    f'\nТекущий лидер: {_best["clusterer"]} + {_best["model"]} '
    f'— val_MAE={_best["val_mae"]:.2f} (Δval={_best["delta_val"]:+.2f})'
)
print(f'{"─" * 80}')

del _interim_df, _interim_clust, _interim_model, _interim_pivot, _interim_plot, _interim_top, _best
gc.collect()

In [ ]:
# ── БАТЧ 4/6: Trompt, ExcelFormer, ARM-Net, NODE ──────────────
_batch_names = DL_BATCHES[4]
print(f'{"═"*80}\n  БАТЧ 4/6: {_batch_names}\n{"═"*80}')
_t0 = time.time()
_batch4_results = []
for _cname, _pgrid in CLUSTER_EXPERIMENTS.items():
    _df = run_one_clusterer_dl(_cname, _pgrid, dl_names=_batch_names, include_grownet=False)
    _batch4_results.append(_df)
dl_cluster_all_results.extend(_batch4_results)
print(f'\nБатч 4 завершён за {time.time()-_t0:.0f}s, строк: {sum(len(d) for d in _batch4_results)}')

# ── Промежуточный анализ после батча 4 ──────────────────────
_interim_df = pd.concat(dl_cluster_all_results, ignore_index=True)
print(f'\n{"═" * 80}')
print(f'  ПРОМЕЖУТОЧНЫЙ АНАЛИЗ ПОСЛЕ БАТЧА 4/6 (накоплено {len(_interim_df)} строк)')
print(f'{"═" * 80}')

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО КЛАСТЕРИЗАТОРАМ:')
_interim_clust = _interim_df.groupby('clusterer')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_clust)

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО DL-МОДЕЛЯМ:')
_interim_model = _interim_df.groupby('model')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_model)

_interim_pivot = _interim_df.pivot_table(index='clusterer', columns='model', values='delta_val', aggfunc='max')
_interim_pivot['avg'] = _interim_pivot.mean(axis=1)
_interim_pivot = _interim_pivot.sort_values('avg', ascending=False)
_interim_plot = _interim_pivot.drop(columns=['avg'])

fig, ax = plt.subplots(figsize=(max(12, len(_interim_plot.columns)*1.2), 8))
sns.heatmap(_interim_plot, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Промежуточная heatmap после батча 4/6: Max Δ val MAE (>0 = улучшение)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

_interim_top = _interim_df.sort_values('delta_val', ascending=False).head(10).reset_index(drop=True)
print(f'\n  ТОП-10 ЛУЧШИХ КОМБИНАЦИЙ (на данный момент):')
display(_interim_top[['clusterer', 'params', 'fit_mode', 'model', 'val_mae', 'baseline_val', 'delta_val']])

_best = _interim_df.loc[_interim_df['delta_val'].idxmax()]
print(
    f'\nТекущий лидер: {_best["clusterer"]} + {_best["model"]} '
    f'— val_MAE={_best["val_mae"]:.2f} (Δval={_best["delta_val"]:+.2f})'
)
print(f'{"─" * 80}')

del _interim_df, _interim_clust, _interim_model, _interim_pivot, _interim_plot, _interim_top, _best
gc.collect()

In [ ]:
# ── БАТЧ 5/6: GRANDE, Net-DNF, TabNet, TabR ───────────────────
_batch_names = DL_BATCHES[5]
print(f'{"═"*80}\n  БАТЧ 5/6: {_batch_names}\n{"═"*80}')
_t0 = time.time()
_batch5_results = []
for _cname, _pgrid in CLUSTER_EXPERIMENTS.items():
    _df = run_one_clusterer_dl(_cname, _pgrid, dl_names=_batch_names, include_grownet=False)
    _batch5_results.append(_df)
dl_cluster_all_results.extend(_batch5_results)
print(f'\nБатч 5 завершён за {time.time()-_t0:.0f}s, строк: {sum(len(d) for d in _batch5_results)}')

# ── Промежуточный анализ после батча 5 ──────────────────────
_interim_df = pd.concat(dl_cluster_all_results, ignore_index=True)
print(f'\n{"═" * 80}')
print(f'  ПРОМЕЖУТОЧНЫЙ АНАЛИЗ ПОСЛЕ БАТЧА 5/6 (накоплено {len(_interim_df)} строк)')
print(f'{"═" * 80}')

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО КЛАСТЕРИЗАТОРАМ:')
_interim_clust = _interim_df.groupby('clusterer')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_clust)

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО DL-МОДЕЛЯМ:')
_interim_model = _interim_df.groupby('model')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_model)

_interim_pivot = _interim_df.pivot_table(index='clusterer', columns='model', values='delta_val', aggfunc='max')
_interim_pivot['avg'] = _interim_pivot.mean(axis=1)
_interim_pivot = _interim_pivot.sort_values('avg', ascending=False)
_interim_plot = _interim_pivot.drop(columns=['avg'])

fig, ax = plt.subplots(figsize=(max(12, len(_interim_plot.columns)*1.2), 8))
sns.heatmap(_interim_plot, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Промежуточная heatmap после батча 5/6: Max Δ val MAE (>0 = улучшение)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

_interim_top = _interim_df.sort_values('delta_val', ascending=False).head(10).reset_index(drop=True)
print(f'\n  ТОП-10 ЛУЧШИХ КОМБИНАЦИЙ (на данный момент):')
display(_interim_top[['clusterer', 'params', 'fit_mode', 'model', 'val_mae', 'baseline_val', 'delta_val']])

_best = _interim_df.loc[_interim_df['delta_val'].idxmax()]
print(
    f'\nТекущий лидер: {_best["clusterer"]} + {_best["model"]} '
    f'— val_MAE={_best["val_mae"]:.2f} (Δval={_best["delta_val"]:+.2f})'
)
print(f'{"─" * 80}')

del _interim_df, _interim_clust, _interim_model, _interim_pivot, _interim_plot, _interim_top, _best
gc.collect()

In [ ]:
# ── БАТЧ 6/6: SwitchTab, PTaRL, DCNv2, GrowNet ────────────────
_batch_names = [n for n in DL_BATCHES[6] if n != "GrowNet"]
print(f'{"═"*80}\n  БАТЧ 6/6: {DL_BATCHES[6]}\n{"═"*80}')
_t0 = time.time()
_batch6_results = []
for _cname, _pgrid in CLUSTER_EXPERIMENTS.items():
    _df = run_one_clusterer_dl(_cname, _pgrid, dl_names=_batch_names, include_grownet=True)
    _batch6_results.append(_df)
dl_cluster_all_results.extend(_batch6_results)
print(f'\nБатч 6 завершён за {time.time()-_t0:.0f}s, строк: {sum(len(d) for d in _batch6_results)}')

# ── Объединяем все батчи ───────────────────────────────────────
dl_cluster_results_df = pd.concat(dl_cluster_all_results, ignore_index=True)
dl_cluster_elapsed = time.time() - _t0
print(f'\nВсего строк: {len(dl_cluster_results_df)} | Моделей: {dl_cluster_results_df["model"].nunique()} | Кластеризаторов: {dl_cluster_results_df["clusterer"].nunique()}')

# ── Промежуточный анализ после батча 6 ──────────────────────
_interim_df = pd.concat(dl_cluster_all_results, ignore_index=True)
print(f'\n{"═" * 80}')
print(f'  ПРОМЕЖУТОЧНЫЙ АНАЛИЗ ПОСЛЕ БАТЧА 6/6 (накоплено {len(_interim_df)} строк)')
print(f'{"═" * 80}')

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО КЛАСТЕРИЗАТОРАМ:')
_interim_clust = _interim_df.groupby('clusterer')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_clust)

print(f'\n  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО DL-МОДЕЛЯМ:')
_interim_model = _interim_df.groupby('model')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(_interim_model)

_interim_pivot = _interim_df.pivot_table(index='clusterer', columns='model', values='delta_val', aggfunc='max')
_interim_pivot['avg'] = _interim_pivot.mean(axis=1)
_interim_pivot = _interim_pivot.sort_values('avg', ascending=False)
_interim_plot = _interim_pivot.drop(columns=['avg'])

fig, ax = plt.subplots(figsize=(max(12, len(_interim_plot.columns)*1.2), 8))
sns.heatmap(_interim_plot, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Промежуточная heatmap после батча 6/6: Max Δ val MAE (>0 = улучшение)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()

_interim_top = _interim_df.sort_values('delta_val', ascending=False).head(10).reset_index(drop=True)
print(f'\n  ТОП-10 ЛУЧШИХ КОМБИНАЦИЙ (на данный момент):')
display(_interim_top[['clusterer', 'params', 'fit_mode', 'model', 'val_mae', 'baseline_val', 'delta_val']])

_best = _interim_df.loc[_interim_df['delta_val'].idxmax()]
print(
    f'\nТекущий лидер: {_best["clusterer"]} + {_best["model"]} '
    f'— val_MAE={_best["val_mae"]:.2f} (Δval={_best["delta_val"]:+.2f})'
)
print(f'{"─" * 80}')

del _interim_df, _interim_clust, _interim_model, _interim_pivot, _interim_plot, _interim_top, _best
gc.collect()

In [ ]:

# ═══════════════════════════════════════════════════════════════
#  АНАЛИЗ SCREENING РЕЗУЛЬТАТОВ: кластеризация × DL (validation only)
# ═══════════════════════════════════════════════════════════════

print(f'\n{"═" * 80}')
print('  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО КЛАСТЕРИЗАТОРАМ (DL)')
print(f'{"═" * 80}')

dl_cluster_summary_df = dl_cluster_results_df.groupby('clusterer')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(dl_cluster_summary_df)

print(f'\n{"═" * 80}')
print('  СРЕДНЯЯ ДЕЛЬТА VAL MAE ПО DL-МОДЕЛЯМ')
print(f'{"═" * 80}')

dl_model_summary = dl_cluster_results_df.groupby('model')['delta_val'].agg(
    ['mean', 'median', 'std', 'max', 'min']
).round(2).sort_values('mean', ascending=False)
display(dl_model_summary)


In [ ]:

# Heatmap: кластеризатор × DL-модель (max validation delta)
dl_cluster_pivot = dl_cluster_results_df.pivot_table(
    index='clusterer',
    columns='model',
    values='delta_val',
    aggfunc='max'
)

dl_cluster_pivot['avg'] = dl_cluster_pivot.mean(axis=1)
dl_cluster_pivot = dl_cluster_pivot.sort_values('avg', ascending=False)
dl_cluster_plot = dl_cluster_pivot.drop(columns=['avg'])

fig, ax = plt.subplots(figsize=(24, 10))
sns.heatmap(dl_cluster_plot, annot=True, fmt='.1f', cmap='RdYlGn',
            center=0, linewidths=0.5, ax=ax)
ax.set_title('Max Δ validation MAE: кластеризатор × DL-модель (>0 = улучшение)')
plt.xticks(rotation=60, ha='right')
plt.tight_layout()
plt.show()


In [ ]:

print(f'\n{"═" * 80}')
print('  ТОП-20 ЛУЧШИХ КОМБИНАЦИЙ (DL × Кластеризация) ПО VALIDATION')
print(f'{"═" * 80}')

dl_cluster_top20 = dl_cluster_results_df.sort_values('delta_val', ascending=False).head(20).reset_index(drop=True)
display(dl_cluster_top20[['clusterer', 'params', 'fit_mode', 'model', 'val_mae', 'baseline_val', 'delta_val']])

dl_cluster_best_row = dl_cluster_results_df.loc[dl_cluster_results_df['delta_val'].idxmax()]
print(f'\nЛучшая комбинация (по validation):')
print(f'  Кластеризатор: {dl_cluster_best_row["clusterer"]} {dl_cluster_best_row["params"]}')
print(f'  DL-модель:     {dl_cluster_best_row["model"]}')
print(f'  val MAE:       {dl_cluster_best_row["val_mae"]:.2f} (baseline {dl_cluster_best_row["baseline_val"]:.2f}, Δ={dl_cluster_best_row["delta_val"]:+.2f})')


In [ ]:

# Barplot: средняя validation delta по кластеризаторам (DL)
dl_cluster_bar = dl_cluster_summary_df['mean'].sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#4CAF50' if v > 0 else '#F44336' for v in dl_cluster_bar.values]

ax.barh(range(len(dl_cluster_bar)), dl_cluster_bar.values, color=colors, edgecolor='white')
ax.set_yticks(range(len(dl_cluster_bar)))
ax.set_yticklabels(dl_cluster_bar.index)
ax.set_xlabel('Средняя Δ validation MAE (>0 = улучшение)')
ax.set_title('Влияние кластеризации на DL-модели (validation delta)')
ax.axvline(0, color='black', linewidth=0.8)
plt.tight_layout()
plt.show()


In [ ]:
# ═══════════════════════════════════════════════════════════════
#  TOP-10 DL × cluster by best validation MAE (CV)
# ═══════════════════════════════════════════════════════════════

best_per_dl = dl_cluster_results_df.loc[
    dl_cluster_results_df.groupby('model')['val_mae'].idxmin()
].sort_values('val_mae', ascending=True).reset_index(drop=True)

top10_dl = best_per_dl.head(10).copy()

holdout_typ_mask = (holdout_raw[target_col] < DURATION_CAP).reset_index(drop=True).values

top10_aug_data = {}

for _, row in top10_dl.iterrows():
    dl_name = row['model']
    clusterer_name = row['clusterer']
    params = ast.literal_eval(row['params'])

    result_full = cluster_fit_and_assign(
        clusterer_name, params, cluster_Xtr_embed, cluster_Xte_embed
    )
    if result_full is None:
        continue

    tr_labels_full, te_labels_full, tr_proba_full, te_proba_full, fit_mode = result_full

    tr_feat_full, te_feat_full = cluster_build_meta_features(
        tr_labels_full, te_labels_full,
        tr_proba=tr_proba_full, te_proba=te_proba_full
    )

    Xtr_aug = pd.concat(
        [cluster_Xtr0.reset_index(drop=True), tr_feat_full.reset_index(drop=True)],
        axis=1
    )
    Xte_full_aug = pd.concat(
        [cluster_Xte0.reset_index(drop=True), te_feat_full.reset_index(drop=True)],
        axis=1
    )

    Xte_typ_aug = Xte_full_aug.loc[holdout_typ_mask].reset_index(drop=True)

    val_cut = int(len(Xtr_aug) * 0.8)
    X_tr_np = Xtr_aug.iloc[:val_cut].values.astype(np.float32)
    y_tr_np = cluster_ytr0[:val_cut].astype(np.float32)
    X_va_np = Xtr_aug.iloc[val_cut:].values.astype(np.float32)
    y_va_np = cluster_ytr0[val_cut:].astype(np.float32)

    X_te_full_np = Xte_full_aug.values.astype(np.float32)
    X_te_typ_np = Xte_typ_aug.values.astype(np.float32)

    sc_aug = StandardScaler()
    X_tr_np = sc_aug.fit_transform(X_tr_np)
    X_va_np = sc_aug.transform(X_va_np)
    X_te_full_np = sc_aug.transform(X_te_full_np)
    X_te_typ_np = sc_aug.transform(X_te_typ_np)

    for arr in [X_tr_np, X_va_np, X_te_full_np, X_te_typ_np]:
        np.nan_to_num(arr, copy=False)

    sc_full = StandardScaler()
    X_full_np = sc_full.fit_transform(Xtr_aug.values.astype(np.float32))
    X_te_full_store_np = sc_full.transform(Xte_full_aug.values.astype(np.float32))
    X_te_typ_store_np = X_te_full_store_np[holdout_typ_mask]

    np.nan_to_num(X_full_np, copy=False)
    np.nan_to_num(X_te_full_store_np, copy=False)
    np.nan_to_num(X_te_typ_store_np, copy=False)

    top10_aug_data[dl_name] = {
        'd_in': Xtr_aug.shape[1],
        'X_tr': torch.from_numpy(X_tr_np),
        'y_tr': torch.from_numpy(y_tr_np).unsqueeze(1),
        'X_va': torch.from_numpy(X_va_np),
        'y_va': torch.from_numpy(y_va_np).unsqueeze(1),
        'X_te_full': torch.from_numpy(X_te_full_np),
        'X_te_typ': torch.from_numpy(X_te_typ_np),
        'X_full': torch.from_numpy(X_full_np),
        'y_full': torch.from_numpy(cluster_ytr0.astype(np.float32)).unsqueeze(1),
        'X_te_full_refit': torch.from_numpy(X_te_full_store_np),
        'X_te_typ_refit': torch.from_numpy(X_te_typ_store_np),
        'clusterer': clusterer_name,
        'cluster_params': params,
        'fit_mode': fit_mode,
        'screening_val_mae': row['val_mae'],
        'screening_delta_val': row['delta_val'],
    }

print(f"\nПодготовлено {len(top10_aug_data)} наборов аугментированных данных.")

In [ ]:
# ═══════════════════════════════════════════════════════════════
#  Optuna-тюнинг TOP-10 DL с лучшей кластеризацией (selected by validation)
# ═══════════════════════════════════════════════════════════════

N_TRIALS_DL_CLUSTER = 50
SEED = 2


def build_dl_cluster_model(dl_name, d_in, params):
    dropout = params.get("dropout", 0.3)

    if dl_name == "MLP":
        n_layers = params["n_layers"]
        first = params["first_hidden"]
        dims = [max(first // (2 ** i), 32) for i in range(n_layers)]
        return MLP(d_in, dims, dropout=dropout)

    elif dl_name == "ResMLP":
        return ResMLP(
            d_in,
            hidden=params["hidden"],
            n_blocks=params["n_blocks"],
            dropout=dropout,
        )

    elif dl_name == "SNN":
        n_layers = params["n_layers"]
        first = params["first_hidden"]
        dims = [max(first // (2 ** i), 32) for i in range(n_layers)]
        return SNN(
            d_in,
            hidden_dims=dims,
            alpha_dropout=params["alpha_dropout"],
        )

    elif dl_name == "GatedMLP":
        return GatedMLP(
            d_in,
            hidden=params["hidden"],
            n_blocks=params["n_blocks"],
            dropout=dropout,
        )

    elif dl_name == "GANDALF":
        return GANDALF(
            d_in,
            hidden=params["hidden"],
            n_blocks=params["n_blocks"],
            dropout=dropout,
        )

    elif dl_name == "DAE-MLP":
        return DAEMLP(
            d_in,
            hidden=params["hidden"],
            latent=params["latent"],
            noise=params["noise"],
            dropout=dropout,
        )

    elif dl_name == "CNN1D":
        n_conv = params["n_conv"]
        base_ch = params["base_ch"]
        chs = [base_ch * (2 ** i) for i in range(n_conv // 2 + 1)]
        chs += [base_ch * (2 ** i) for i in range(n_conv // 2 - 1, -1, -1)]
        chs = chs[:n_conv]
        return CNN1D(
            d_in,
            channels=chs,
            ks=params["ks"],
            dropout=dropout,
        )

    elif dl_name == "BiGRU":
        return BiGRU(
            d_in,
            hidden=params["hidden"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "FT-Transformer":
        return FTTransformer(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "TabTransformer":
        return TabTransformer(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            mlp_hidden=params["mlp_hidden"],
            dropout=dropout,
        )

    elif dl_name == "SAINT":
        return SAINT(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "AutoInt":
        return AutoInt(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "Trompt":
        return Trompt(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "ExcelFormer":
        return ExcelFormer(
            d_in,
            d_model=params["d_model"],
            n_heads=params["n_heads"],
            n_layers=params["n_layers"],
            dropout=dropout,
        )

    elif dl_name == "ARM-Net":
        return ARMNet(
            d_in,
            n_exp=params["n_exp"],
            hidden=params["hidden"],
            order=params["order"],
            dropout=dropout,
        )

    elif dl_name == "NODE":
        return NODE(
            d_in,
            n_trees=params["num_trees"],
            depth=params["depth"],
            dropout=dropout,
        )

    elif dl_name == "GRANDE":
        return GRANDE(
            d_in,
            n_trees=params["num_trees"],
            depth=params["depth"],
            dropout=dropout,
        )

    elif dl_name == "Net-DNF":
        return NetDNF(
            d_in,
            n_formulas=params["n_formulas"],
            n_conjuncts=params["n_conjuncts"],
            dropout=dropout,
        )

    elif dl_name == "TabNet":
        return TabNet(
            d_in,
            n_steps=params["n_steps"],
            n_d=params["n_d"],
            n_a=params["n_a"],
            dropout=dropout,
        )

    elif dl_name == "TabR":
        return TabR(
            d_in,
            hidden=params["hidden"],
            n_layers=params["n_layers"],
            k=params["k"],
            dropout=dropout,
        )

    elif dl_name == "SwitchTab":
        return SwitchTab(
            d_in,
            d_enc=params["d_enc"],
            d_mutual=params["d_mutual"],
            d_salient=params["d_salient"],
            dropout=dropout,
        )

    elif dl_name == "PTaRL":
        return PTaRL(
            d_in,
            n_prototypes=params["n_prototypes"],
            d_latent=params["d_latent"],
            hidden=params["hidden"],
            dropout=dropout,
        )

    elif dl_name == "DCNv2":
        return DCNv2(
            d_in,
            n_cross=params["n_cross"],
            hidden=params["hidden"],
            dropout=dropout,
        )

    else:
        raise ValueError(dl_name)


def make_dl_cluster_objective(dl_name, data):
    d_in = data['d_in']
    X_tr, y_tr = data['X_tr'], data['y_tr']
    X_va, y_va = data['X_va'], data['y_va']

    def objective(trial):
        params = {
            "lr": trial.suggest_float("lr", 1e-4, 3e-3, log=True),
            "wd": trial.suggest_float("wd", 1e-5, 1e-3, log=True),
            "bs": trial.suggest_categorical("bs", [128, 256, 512]),
            "dropout": trial.suggest_float("dropout", 0.1, 0.5),
        }

        if dl_name == "MLP":
            params["n_layers"] = trial.suggest_int("n_layers", 2, 4)
            params["first_hidden"] = trial.suggest_categorical("first_hidden", [128, 256, 512])

        elif dl_name == "ResMLP":
            params["hidden"] = trial.suggest_categorical("hidden", [128, 256, 384])
            params["n_blocks"] = trial.suggest_int("n_blocks", 2, 5)

        elif dl_name == "SNN":
            params["n_layers"] = trial.suggest_int("n_layers", 2, 4)
            params["first_hidden"] = trial.suggest_categorical("first_hidden", [128, 256, 512])
            params["alpha_dropout"] = trial.suggest_float("alpha_dropout", 0.01, 0.2)

        elif dl_name == "GatedMLP":
            params["hidden"] = trial.suggest_categorical("hidden", [128, 256, 384])
            params["n_blocks"] = trial.suggest_int("n_blocks", 2, 5)

        elif dl_name == "GANDALF":
            params["hidden"] = trial.suggest_categorical("hidden", [128, 256, 384])
            params["n_blocks"] = trial.suggest_int("n_blocks", 2, 5)

        elif dl_name == "DAE-MLP":
            params["hidden"] = trial.suggest_categorical("hidden", [128, 256, 512])
            params["latent"] = trial.suggest_categorical("latent", [32, 64, 128])
            params["noise"] = trial.suggest_float("noise", 0.1, 0.5)

        elif dl_name == "CNN1D":
            params["n_conv"] = trial.suggest_int("n_conv", 2, 4)
            params["base_ch"] = trial.suggest_categorical("base_ch", [32, 64, 128])
            params["ks"] = trial.suggest_categorical("ks", [3, 5, 7])

        elif dl_name == "BiGRU":
            params["hidden"] = trial.suggest_categorical("hidden", [32, 64, 128])
            params["n_layers"] = trial.suggest_int("n_layers", 1, 3)

        elif dl_name == "FT-Transformer":
            params["d_model"] = trial.suggest_categorical("d_model", [16, 32, 64])
            params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4, 8])
            params["n_layers"] = trial.suggest_int("n_layers", 1, 3)

        elif dl_name == "TabTransformer":
            params["d_model"] = trial.suggest_categorical("d_model", [16, 32, 64])
            params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4, 8])
            params["n_layers"] = trial.suggest_int("n_layers", 1, 3)
            params["mlp_hidden"] = trial.suggest_categorical("mlp_hidden", [64, 128, 256])

        elif dl_name == "SAINT":
            params["d_model"] = trial.suggest_categorical("d_model", [16, 32, 64])
            params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4])
            params["n_layers"] = trial.suggest_int("n_layers", 1, 3)

        elif dl_name == "AutoInt":
            params["d_model"] = trial.suggest_categorical("d_model", [16, 32, 64])
            params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4, 8])
            params["n_layers"] = trial.suggest_int("n_layers", 2, 4)

        elif dl_name == "Trompt":
            params["d_model"] = trial.suggest_categorical("d_model", [16, 32, 64])
            params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4])
            params["n_layers"] = trial.suggest_int("n_layers", 2, 4)

        elif dl_name == "ExcelFormer":
            params["d_model"] = trial.suggest_categorical("d_model", [16, 32, 64])
            params["n_heads"] = trial.suggest_categorical("n_heads", [2, 4, 8])
            params["n_layers"] = trial.suggest_int("n_layers", 1, 3)

        elif dl_name == "ARM-Net":
            params["n_exp"] = trial.suggest_categorical("n_exp", [32, 64, 128])
            params["hidden"] = trial.suggest_categorical("hidden", [64, 128, 256])
            params["order"] = trial.suggest_int("order", 2, 3)

        elif dl_name == "NODE":
            params["num_trees"] = trial.suggest_categorical("num_trees", [16, 32, 64])
            params["depth"] = trial.suggest_int("depth", 4, 8)

        elif dl_name == "GRANDE":
            params["num_trees"] = trial.suggest_categorical("num_trees", [16, 32, 64])
            params["depth"] = trial.suggest_int("depth", 2, 4)

        elif dl_name == "Net-DNF":
            params["n_formulas"] = trial.suggest_categorical("n_formulas", [32, 64, 128])
            params["n_conjuncts"] = trial.suggest_categorical("n_conjuncts", [4, 6, 8])

        elif dl_name == "TabNet":
            params["n_d"] = trial.suggest_categorical("n_d", [16, 32, 64])
            params["n_a"] = trial.suggest_categorical("n_a", [16, 32, 64])
            params["n_steps"] = trial.suggest_int("n_steps", 3, 5)

        elif dl_name == "TabR":
            params["hidden"] = trial.suggest_categorical("hidden", [128, 256, 512])
            params["n_layers"] = trial.suggest_int("n_layers", 2, 4)
            params["k"] = trial.suggest_categorical("k", [16, 32, 64])

        elif dl_name == "SwitchTab":
            params["d_enc"] = trial.suggest_categorical("d_enc", [128, 256, 384])
            params["d_mutual"] = trial.suggest_categorical("d_mutual", [32, 64, 128])
            params["d_salient"] = trial.suggest_categorical("d_salient", [32, 64, 128])

        elif dl_name == "PTaRL":
            params["hidden"] = trial.suggest_categorical("hidden", [128, 256, 384])
            params["d_latent"] = trial.suggest_categorical("d_latent", [64, 128, 256])
            params["n_prototypes"] = trial.suggest_categorical("n_prototypes", [8, 16, 32])

        elif dl_name == "DCNv2":
            params["hidden"] = trial.suggest_categorical("hidden", [128, 256, 384])
            params["n_cross"] = trial.suggest_int("n_cross", 2, 4)

        elif dl_name == "GrowNet":
            params["hidden"] = trial.suggest_categorical("hidden", [16, 32, 64])
            params["n_stages"] = trial.suggest_int("n_stages", 3, 8)
            params["step_size"] = trial.suggest_float("step_size", 0.03, 0.2)

            torch.manual_seed(SEED)
            _, best_val = train_grownet_aug(
                X_tr, y_tr, X_va, y_va, X_va,
                n_stages=params["n_stages"],
                hidden=params["hidden"],
                lr=params["lr"],
                wd=params["wd"],
                step_size=params["step_size"],
                bs=params["bs"],
                patience=30,
            )
            return best_val

        else:
            raise ValueError(dl_name)

        aux = DL_AUX_W.get(dl_name, 0.0)
        torch.manual_seed(SEED)
        model = build_dl_cluster_model(dl_name, d_in, params)
        model, best_val, _ = train_model_aug(
            model, X_tr, y_tr, X_va, y_va,
            epochs=300,
            lr=params["lr"],
            wd=params["wd"],
            bs=params["bs"],
            patience=30,
            aux_w=aux,
        )
        return best_val

    return objective


dl_cluster_optuna_rows = []

for dl_name, data in top10_aug_data.items():
    print(f'\n{"═"*90}')
    print(f'  Optuna tuning: {dl_name} | cluster = {data["clusterer"]} {data["cluster_params"]}')
    print(f'{"═"*90}')
    t0 = time.time()

    study = optuna.create_study(
        direction="minimize",
        sampler=optuna.samplers.TPESampler(
            seed=SEED,
            multivariate=True,
            warn_independent_sampling=False
        ),
        pruner=optuna.pruners.MedianPruner(
            n_startup_trials=min(8, max(3, N_TRIALS_DL_CLUSTER // 3)),
            n_warmup_steps=20
        ),
    )
    study.optimize(
        make_dl_cluster_objective(dl_name, data),
        n_trials=N_TRIALS_DL_CLUSTER,
        show_progress_bar=False
    )

    best_trial = study.best_trial
    params = dict(best_trial.params)

    lr = params.get("lr", 1e-3)
    wd = params.get("wd", 1e-4)
    bs = params.get("bs", 256)

    d_in = data['d_in']
    aux = DL_AUX_W.get(dl_name, 0.0)

    # Retrain tuned architecture on inner train/validation split; evaluate on external holdout.
    torch.manual_seed(SEED)

    if dl_name == "GrowNet":
        gn_stages, _ = train_grownet_aug(
            data['X_tr'], data['y_tr'], data['X_va'], data['y_va'], data['X_te_full'],
            n_stages=params["n_stages"],
            hidden=params["hidden"],
            lr=lr,
            wd=wd,
            step_size=params["step_size"],
            bs=bs,
            patience=30,
        )

        with torch.no_grad():
            p_full = torch.zeros(data['X_te_full'].size(0), 1, device=device)
            for s in gn_stages:
                s.eval()
                p_full = p_full + params["step_size"] * s(data['X_te_full'].to(device), p_full)
            pred_full = p_full.cpu().numpy().flatten()

            p_typ = torch.zeros(data['X_te_typ'].size(0), 1, device=device)
            for s in gn_stages:
                s.eval()
                p_typ = p_typ + params["step_size"] * s(data['X_te_typ'].to(device), p_typ)
            pred_typ = p_typ.cpu().numpy().flatten()

    else:
        model = build_dl_cluster_model(dl_name, d_in, params)
        model, _, _ = train_model_aug(
            model, data['X_tr'], data['y_tr'], data['X_va'], data['y_va'],
            epochs=300,
            lr=lr,
            wd=wd,
            bs=bs,
            patience=30,
            aux_w=aux,
        )

        if dl_name == "TabR":
            X_store = torch.cat([data['X_tr'], data['X_va']], dim=0)
            y_store = torch.cat([data['y_tr'], data['y_va']], dim=0)
            model.build_store(X_store.to(device), y_store.to(device))

        hold_model = model
        hold_model.eval()

        with torch.no_grad():
            pred_full = hold_model(data['X_te_full'].to(device)).cpu().numpy().flatten()
            pred_typ = hold_model(data['X_te_typ'].to(device)).cpu().numpy().flatten()

    mae_full = mean_absolute_error(cluster_yte0, pred_full)
    rmse_full = np.sqrt(mean_squared_error(cluster_yte0, pred_full))
    mdae_full = median_absolute_error(cluster_yte0, pred_full)

    mae_typ = mean_absolute_error(cluster_ytyp0, pred_typ)
    rmse_typ = np.sqrt(mean_squared_error(cluster_ytyp0, pred_typ))
    mdae_typ = median_absolute_error(cluster_ytyp0, pred_typ)

    baseline_full = dl_baseline_holdout_full_mae.get(dl_name, mae_full)
    baseline_typ = dl_baseline_holdout_typ_mae.get(dl_name, mae_typ)
    elapsed = time.time() - t0

    dl_cluster_optuna_rows.append({
        "model": dl_name,
        "clusterer": data['clusterer'],
        "cluster_params": str(data['cluster_params']),
        "screening_val_MAE": round(data['screening_val_mae'], 2),
        "screening_delta_val": round(data['screening_delta_val'], 2),
        "baseline_holdout_typical_MAE": round(baseline_typ, 2),
        "baseline_holdout_full_MAE": round(baseline_full, 2),
        "tuned_cv_MAE": round(study.best_value, 2),
        "tuned_holdout_typical_MAE": round(mae_typ, 2),
        "tuned_holdout_typical_RMSE": round(rmse_typ, 2),
        "tuned_holdout_typical_MdAE": round(mdae_typ, 2),
        "tuned_holdout_full_MAE": round(mae_full, 2),
        "tuned_holdout_full_RMSE": round(rmse_full, 2),
        "tuned_holdout_full_MdAE": round(mdae_full, 2),
        "delta_vs_baseline_typical": round(baseline_typ - mae_typ, 2),
        "delta_vs_baseline_full": round(baseline_full - mae_full, 2),
        "best_params": params,
        "time_sec": round(elapsed, 1),
    })

    print(f"  Baseline holdout typ/full:  {baseline_typ:.2f} / {baseline_full:.2f}")
    print(f"  Best CV MAE:                {study.best_value:.2f}")
    print(f"  Tuned holdout typ/full:     {mae_typ:.2f} / {mae_full:.2f}")
    print(f"  Best params: {params}")
    print(f"  Time: {elapsed:.1f}s")

    gc.collect()
    if device.type == "mps":
        torch.mps.empty_cache()

dl_cluster_optuna_df = (
    pd.DataFrame(dl_cluster_optuna_rows)
      .sort_values("tuned_holdout_full_MAE")
      .reset_index(drop=True)
)
display(dl_cluster_optuna_df)

In [ ]:
import json
# Save main results
os.makedirs(ARTIFACT_DIR, exist_ok=True)

try:
    dl_baseline_df.to_csv(f"{ARTIFACT_DIR}/dl_cluster_baseline_all24_final_clean.csv", index=False)
    print("Сохранено: dl_cluster_baseline_all24_final_clean.csv")
except Exception as e:
    print("Ошибка сохранения baseline:", e)

try:
    dl_cluster_results_df.to_csv(f"{ARTIFACT_DIR}/dl_cluster_screening_all24_final_clean.csv", index=False)
    print("Сохранено: dl_cluster_screening_all24_final_clean.csv")
except Exception as e:
    print("Ошибка сохранения screening:", e)

try:
    dl_cluster_summary_df.to_csv(f"{ARTIFACT_DIR}/dl_cluster_summary_all24_final_clean.csv")
    print("Сохранено: dl_cluster_summary_all24_final_clean.csv")
except Exception as e:
    print("Ошибка сохранения summary:", e)

try:
    best_per_dl.to_csv(f"{ARTIFACT_DIR}/dl_cluster_best_by_model_all24_final_clean.csv", index=False)
    print("Сохранено: dl_cluster_best_by_model_all24_final_clean.csv")
except Exception as e:
    print("Ошибка сохранения best-by-model:", e)

try:
    dl_cluster_optuna_df.to_csv(f"{ARTIFACT_DIR}/dl_cluster_tuned_top10_by_cv_all24_final_clean.csv", index=False)
    print("Сохранено: dl_cluster_tuned_top10_by_cv_all24_final_clean.csv")
except Exception as e:
    print("Ошибка сохранения tuned:", e)

protocol = {
    "notebook": "DL_cluster",
    "selection_rule": "cluster screening and top-5 selection use inner temporal validation only",
    "external_holdout": "same 80/20 chronological holdout, used only for final evaluation",
    "train_definition": "train_raw filtered to duration_hours < 960",
    "holdout_typical_definition": "holdout_raw filtered to duration_hours < 960",
    "holdout_full_definition": "full external holdout",
    "n_trials_dl_cluster": N_TRIALS_DL_CLUSTER,
}
with open(f"{ARTIFACT_DIR}/dl_cluster_protocol_all24_final_clean.json", "w", encoding="utf-8") as f:
    json.dump(protocol, f, ensure_ascii=False, indent=2)
print("Сохранено: dl_cluster_protocol_all24_final_clean.json")
